In [1]:
# 📦 Imports
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
from sklearn.utils import resample
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy.stats import entropy, wasserstein_distance
from VAE_model import VAE
import torch
# 🔹 Display plots inline
%matplotlib inline


AttributeError: 'RcParams' object has no attribute '_get'

In [ ]:
# Load original from h5ad
adata = sc.read_h5ad("data/tabula_muris/all.h5ad")
X_real = adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X
print("✅ Real data shape:", X_real.shape)

# Load generated data from npz
gen = np.load("output/simulated_samples/breastcancerGenerated.npz")
X_gen = gen["cell_gen"]
print("✅ Generated data shape:", X_gen.shape)


✅ Real data shape: (42, 22283)
✅ Generated data shape: (50, 128)


In [11]:
n = min(len(X_real), len(X_gen))
X_real = resample(X_real, n_samples=n, random_state=42)
X_gen = resample(X_gen, n_samples=n, random_state=42)
print("🔄 Sample size after matching:", X_real.shape, X_gen.shape)


🔄 Sample size after matching: (42, 22283) (42, 128)


In [13]:
data = np.load("output/simulated_samples/breastcancerGenerated.npz")
latent_gen = data["cell_gen"]
print("Generated latent shape:", latent_gen.shape)

# Load trained VAE and decode
vae_ckpt = "output/vae_checkpoints/BREASTCANCER/model_seed=0_step=199999.pt"
vae = VAE(num_genes=22283, device="cpu", hidden_dim=128)
vae.load_state_dict(torch.load(vae_ckpt, map_location="cpu"))
vae.eval()

with torch.no_grad():
    decoded_gen = vae(torch.tensor(latent_gen).float(), return_decoded=True).numpy()

Generated latent shape: (50, 128)


/tmp/ipykernel_1036828/1364138858.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  vae.load_state_dict(torch.load(vae_ckpt, map_location="cpu"))


In [14]:
from scipy.stats import entropy, wasserstein_distance
import numpy as np

# Convert real expression matrix to dense (if needed)
real_expr = adata.X.toarray() if not isinstance(adata.X, np.ndarray) else adata.X

# Match the number of samples
min_samples = min(real_expr.shape[0], decoded_gen.shape[0])
real_subset = real_expr[:min_samples]
gen_subset = decoded_gen[:min_samples]

print("Real subset shape:", real_subset.shape)
print("Generated subset shape:", gen_subset.shape)

# Initialize lists
kl_list = []
wd_list = []

# Compute per-feature similarity
for i in range(real_subset.shape[1]):
    real_col = real_subset[:, i]
    gen_col = gen_subset[:, i]

    # Histogram-based estimate of distributions
    p_hist, _ = np.histogram(real_col, bins=50, density=True)
    q_hist, _ = np.histogram(gen_col, bins=50, density=True)

    # Add smoothing to avoid log(0)
    kl = entropy(p_hist + 1e-10, q_hist + 1e-10)
    wd = wasserstein_distance(real_col, gen_col)

    kl_list.append(kl)
    wd_list.append(wd)

# Print average results
print(f"\n✅ Average KL Divergence per gene: {np.mean(kl_list):.6f}")
print(f"✅ Average Wasserstein Distance per gene: {np.mean(wd_list):.6f}")


Real subset shape: (42, 22283)
Generated subset shape: (42, 22283)

✅ Average KL Divergence per gene: 22.132286
✅ Average Wasserstein Distance per gene: 3.670517
